# 벡터화(Vectorization)로 정형 데이터 가공

`소집교육/dataset/정형` 폴더의 **CSV 3개**를 읽고, 같은 폴더의 **Excel 1개**(`soldier_assignment_dataset`)를 네 번째 표로 읽어 **분석·모델링용 파생변수**를 만듭니다.

- **벡터화**: 행 단위 `for` 루프 대신 **열 전체(배열)에 한 번에** 연산을 적용해 속도와 가독성을 확보합니다.
- **도구**: `pandas` 스칼라 연산 / `numpy` ufunc / 축별 집계(`mean(axis=1)` 등).

> **참고**: 현재 `정형` 폴더에 있는 **CSV는 3개**입니다. 요청하신 「4개의 CSV」에 맞추려면 동일 폴더에 CSV를 하나 더 두거나, 아래 Excel 구간을 `read_csv`로 바꾸면 됩니다.


## 목차

1. 경로 설정 및 4개 파일 적재  
2. 군수품 자동청구 CSV — 재고·소요 기반 파생변수  
3. 서버 유지보수 CSV — 리소스·이슈 파생변수  
4. 장병 복지시설 이용 CSV — 접근성·만족도 파생변수  
5. 인력 적성 데이터 Excel — 능력치 표준화·종합지수  
6. (부록) 루프 vs 벡터화 시간 비교


In [1]:
from __future__ import annotations

import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# 1. 데이터 디렉터리 경로 자동 해결 함수
# -------------------------------------------------------------------------
def resolve_정형_dir() -> Path:
    """
    작업 환경(로컬, 서버, 특정 폴더 내부 등)에 관계없이 
    'dataset/정형' 폴더를 찾아 Path 객체로 반환합니다.
    """
    cwd = Path.cwd().resolve()
    
    # 탐색 후보 리스트: 프로젝트 루트, 상위 폴더, 현재 폴더 기준
    candidates = [
        cwd / "소집교육" / "dataset" / "정형",
        cwd.parent / "dataset" / "정형",
        cwd / "dataset" / "정형",
    ]
    
    for p in candidates:
        # 폴더가 존재하고, 내부에 .csv 파일이 하나라도 있는 경로를 선택
        if p.is_dir() and any(p.glob("*.csv")):
            return p
            
    raise FileNotFoundError(
        "정형 데이터 폴더를 찾지 못했습니다. 작업 디렉터리를 저장소 루트 또는 소집교육/0423 로 두세요."
    )

# -------------------------------------------------------------------------
# 2. 키워드 기반 파일 선택 함수 (OS 간 한글 자모 분리 대응)
# -------------------------------------------------------------------------
def pick_csv(keyword: str) -> Path:
    """
    macOS(NFD)와 Windows/Linux(NFC)의 한글 인코딩 차이를 무시하고
    파일명에 특정 키워드가 포함된 CSV 파일을 찾아 반환합니다.
    """
    # 검색 키워드를 NFC(표준 한글)로 정규화
    kw = unicodedata.normalize("NFC", keyword)
    
    # 정형 데이터 폴더 내의 모든 csv를 탐색
    for p in sorted(정형_dir.glob("*.csv")):
        # 파일명 또한 NFC로 정규화하여 비교 (자모 분리 현상 방지)
        name = unicodedata.normalize("NFC", p.name)
        if kw in name:
            return p
            
    raise FileNotFoundError(f"키워드 {keyword!r}에 해당하는 CSV가 없습니다.")

# -------------------------------------------------------------------------
# 3. 실행 및 파일 정보 확인
# -------------------------------------------------------------------------
# 경로 설정
정형_dir = resolve_정형_dir()
print(f"✅ 데이터 디렉터리 확정: {정형_dir}")

# 각 키워드에 해당하는 파일 경로 할당
p_mil = pick_csv("군수")  # 군수 데이터
p_srv = pick_csv("서버")  # 서버 데이터
p_wel = pick_csv("복지")  # 복지 데이터
# 엑셀 파일은 패턴 매칭으로 첫 번째 파일을 가져옴
p_xlsx = next(정형_dir.glob("soldier_assignment_dataset*.xlsx"))

# 결과 출력 및 용량 체크
print("-" * 50)
for p in (p_mil, p_srv, p_wel, p_xlsx):
    # 파일 크기를 MB 단위로 계산하여 출력
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"{p.name:<30} → {size_mb:.2f} MB")
print("-" * 50)

✅ 데이터 디렉터리 확정: C:\2026\mli_dev_2026\소집교육\dataset\정형
--------------------------------------------------
군수품자동청구_보정데이터(0512).csv        → 0.90 MB
서버유지보수_보정데이터(0512).csv         → 1.01 MB
장병복지시설이용_보완데이터(0512).csv       → 1.59 MB
soldier_assignment_dataset(0512).xlsx → 1.27 MB
--------------------------------------------------


## 1. 군수품 자동청구 CSV

**파생변수 예시**: 재고가 며칠치인지(런웨이), 납기 동안 예상 소모량, 주말 가중 소요 등 — 모두 **열 단위** 연산.


In [2]:
# -------------------------------------------------------------------------
# 1. 파일 적재 (CSV to DataFrame)
# -------------------------------------------------------------------------
mil = pd.read_csv(p_mil)

# -------------------------------------------------------------------------
# 2. 데이터 타입 최적화 (수치형 변환)
# -------------------------------------------------------------------------
# 분석 및 벡터 연산에 사용될 주요 수치형 컬럼 리스트
num_cols = [
    "stock_level",          # 현재 재고량
    "avg_daily_usage",      # 일평균 사용량
    "weekend_usage_ratio",  # 주말 사용 비중
    "delivery_lead_time",   # 배송 리드 타임 (주문 후 도착까지 걸리는 시간)
    "requested_quantity",   # 요청 수량
    "reorder_threshold",    # 재발주 기준점
]

for c in num_cols:
    if c in mil.columns:
        # 데이터에 포함된 비숫자형(문자열 등) 오류를 NaN으로 처리하며 수치형 변환
        mil[c] = pd.to_numeric(mil[c], errors="coerce")

# -------------------------------------------------------------------------
# 3. 벡터화 연산을 통한 분석용 파생 변수 생성
# -------------------------------------------------------------------------
eps = 1e-6  # Zero Division(0으로 나누기) 에러 방지를 위한 미세값

# 일평균 사용량이 0인 경우를 대비해 최소값을 eps로 제한
usage = mil["avg_daily_usage"].clip(lower=eps)

# [재고 버티기 일수] 현재 재고로 며칠을 버틸 수 있는지 계산 (재고 / 사용량)
mil["stock_runway_days"] = (mil["stock_level"] / usage).replace([np.inf, -np.inf], np.nan)

# [리드 타임 내 예상 수요] 물건을 주문하고 받을 때까지 소비될 것으로 예상되는 양
mil["expected_demand_in_lead_time"] = mil["avg_daily_usage"] * mil["delivery_lead_time"]

# [주말 보정 일일 사용량] 주말 사용 비중을 반영한 보정된 일일 사용량
mil["weekend_adjusted_daily"] = mil["avg_daily_usage"] * (1.0 + mil["weekend_usage_ratio"].fillna(0))

# [재발주 갭] 현재 재고가 재발주 기준점에 얼마나 미달하는지 계산
mil["reorder_gap"] = mil["reorder_threshold"] - mil["stock_level"]

# [요청량 대 수요 차이] 요청한 수량이 리드 타임 내 예상 수요를 충당할 수 있는지 확인
mil["request_vs_lead_demand"] = mil["requested_quantity"] - mil["expected_demand_in_lead_time"]

# -------------------------------------------------------------------------
# 4. 시계열 데이터 처리
# -------------------------------------------------------------------------
if "request_date" in mil.columns:
    # 날짜 형식으로 변환
    mil["request_dt"] = pd.to_datetime(mil["request_date"], errors="coerce")
    # 월(Month) 및 요일(Weekday, 0=월요일) 정보 추출 (계절성/주간 패턴 분석용)
    mil["request_month"] = mil["request_dt"].dt.month
    mil["request_weekday"] = mil["request_dt"].dt.weekday

# -------------------------------------------------------------------------
# 5. 주요 지표 요약 통계 확인
# -------------------------------------------------------------------------
# 재고 수준과 생성된 파생 변수들의 분포(평균, 최소/최대값 등) 확인
mil[["stock_level", "stock_runway_days", "expected_demand_in_lead_time", "weekend_adjusted_daily"]].describe()

,stock_level,stock_runway_days,expected_demand_in_lead_time,weekend_adjusted_daily
count,11000.000000,11000.000000,11000.000000,11000.000000
mean,23.224662,5.512729,20.721024,6.279319
std,10.175750,4.206894,12.696916,2.989741
min,2.000000,0.429775,1.000000,0.550000
25%,16.000000,4.754098,11.000000,4.250000
50%,23.000000,4.814815,18.500000,6.032539
75%,30.000000,4.883721,28.000000,7.935000
max,59.000000,46.800864,83.440766,17.738253


## 2. 서버 유지보수 CSV

**파생변수**: CPU·메모리·디스크 평균 부하, 이슈·조치 복합 지표, 조치 시간 로그 스케일 등.


In [3]:
# ── 파일 적재 ─────────────────────────────────────────────────────────
# CSV·Excel을 DataFrame으로 읽기

srv = pd.read_csv(p_srv)
for c in ["cpu_usage", "memory_usage", "disk_usage", "fix_duration_hours"]:
    if c in srv.columns:
        srv[c] = pd.to_numeric(srv[c], errors="coerce")

srv["resource_load_mean"] = srv[["cpu_usage", "memory_usage", "disk_usage"]].mean(axis=1, skipna=True)
srv["resource_load_std"] = srv[["cpu_usage", "memory_usage", "disk_usage"]].std(axis=1, skipna=True)
srv["fix_required"] = pd.to_numeric(srv["fix_required"], errors="coerce").astype("Int64")
srv["issue_detected"] = pd.to_numeric(srv["issue_detected"], errors="coerce").astype("Int64")
srv["incident_workload"] = (
    srv["issue_detected"].fillna(0).astype(float) * srv["fix_required"].fillna(0).astype(float)
)
srv["log1p_fix_hours"] = np.log1p(srv["fix_duration_hours"].clip(lower=0).fillna(0))

# 범주형은 정수 라벨로 (모델 입력용) — 벡터화된 factorize
if "issue_category" in srv.columns:
    codes, uniques = pd.factorize(srv["issue_category"], sort=True)
    srv["issue_category_code"] = codes

srv[["cpu_usage", "resource_load_mean", "fix_duration_hours", "log1p_fix_hours", "incident_workload"]].describe()


,cpu_usage,resource_load_mean,fix_duration_hours,log1p_fix_hours,incident_workload
count,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000
mean,28.866911,30.783633,2.999473,1.318021,0.132182
std,15.373598,9.303116,1.517592,0.369655,0.338704
min,0.000000,10.300000,0.140000,0.131028,0.000000
25%,19.500000,25.200000,1.897500,1.063847,0.000000
50%,25.500000,29.300000,2.750000,1.321756,0.000000
75%,33.600000,34.200000,3.830000,1.574846,0.000000
max,100.000000,86.156950,11.790000,2.548664,1.000000


## 3. 장병 복지시설 이용 CSV

**파생변수**: 거리·이용빈도 기반 부담 지수, 만족도와 정신건강 점수의 간단한 복합 지표 등.


In [4]:
# -------------------------------------------------------------------------
# 1. 파일 적재 및 수치형 변환
# -------------------------------------------------------------------------
wel = pd.read_csv(p_wel)

# 수치형 변환이 필요한 주요 컬럼 정의
target_cols = [
    "facility_usage",           # 시설 이용 횟수
    "usage_frequency_per_week", # 주당 이용 빈도
    "satisfaction_score",       # 만족도 점수
    "distance_to_facility",     # 시설까지의 거리
    "mental_wellbeing_score",   # 정신건강 점수
    "facility_access_score",    # 시설 접근성 점수
    "expected_mental_score",    # 기대 정신건강 점수
]

for c in target_cols:
    if c in wel.columns:
        # 비수치 데이터(문자열 등)를 NaN으로 처리하며 숫자형으로 변환
        wel[c] = pd.to_numeric(wel[c], errors="coerce")

# -------------------------------------------------------------------------
# 2. 복지 체감 지표(Index) 파생 변수 생성
# -------------------------------------------------------------------------
# [이용 부담 지수] 거리가 멀수록, 이용 빈도가 낮을수록 부담이 큼을 의미
# 분모가 0이 되는 것을 방지하기 위해 빈도에 +1.0 처리 (Smoothing)
freq = wel["usage_frequency_per_week"].clip(lower=0) + 1.0
wel["access_burden_index"] = wel["distance_to_facility"] / freq

# [참여도 원천 점수] 시설 이용 경험과 주당 빈도를 곱해 활동성 계산
wel["engagement_raw"] = wel["facility_usage"].fillna(0) * wel["usage_frequency_per_week"].fillna(0)

# -------------------------------------------------------------------------
# 3. 효율적인 Min-Max 정규화 함수 (Numpy 활용)
# -------------------------------------------------------------------------
def min_max_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    선택한 컬럼들을 0~1 사이 값으로 정규화하여 새로운 DataFrame으로 반환.
    Numpy 브로드캐스팅을 사용하여 Pandas 연산보다 빠름.
    """
    x = df[cols].to_numpy(dtype=float)
    mn = np.nanmin(x, axis=0) # 컬럼별 최소값
    mx = np.nanmax(x, axis=0) # 컬럼별 최대값
    
    # 분모가 0이 되는 경우(최대값=최소값) 1.0으로 대체하여 Zero Division 방지
    denom = np.where((mx - mn) < 1e-9, 1.0, mx - mn)
    z = (x - mn) / denom
    
    return pd.DataFrame(z, columns=[f"{c}_01" for c in cols], index=df.index)

# 정규화 적용 및 기존 데이터프레임에 결합 (concat)
mm = min_max_cols(wel, ["distance_to_facility", "satisfaction_score", "mental_wellbeing_score"])
wel = pd.concat([wel, mm], axis=1)

# -------------------------------------------------------------------------
# 4. 벡터화 기반 조건부 플래그(Flag) 생성
# -------------------------------------------------------------------------
# [원거리 저만족군] 거리는 중앙값보다 멀고, 만족도는 중앙값보다 낮은 취약 계층 탐지
# .astype(int)를 통해 True/False를 1/0으로 변환
wel["far_and_low_sat"] = (
    (wel["distance_to_facility"] > wel["distance_to_facility"].median())
    & (wel["satisfaction_score"] <= wel["satisfaction_score"].median())
).astype(int)

# -------------------------------------------------------------------------
# 5. 주요 결과 요약 통계 확인
# -------------------------------------------------------------------------
# 파생 지표와 정규화된 점수, 플래그의 분포 확인
wel[["access_burden_index", "engagement_raw", "satisfaction_score_01", "far_and_low_sat"]].describe()

,access_burden_index,engagement_raw,satisfaction_score_01,far_and_low_sat
count,11000.000000,11000.000000,11000.000000,11000.000000
mean,105.148388,1.233909,0.402879,0.258273
std,75.431804,1.522064,0.258927,0.437705
min,14.528571,0.000000,0.000000,0.000000
25%,57.231250,0.000000,0.166667,0.000000
50%,87.112500,0.000000,0.333333,0.000000
75%,124.833333,2.000000,0.666667,1.000000
max,498.900000,6.000000,1.000000,1.000000


## 4. 인력 적성·배치 Excel (네 번째 정형 표)

능력·적성 수치 열을 **표준화(z-score)** 한 뒤, 행별 평균으로 **종합 역량 지수**를 벡터화로 계산합니다.


In [5]:
# -------------------------------------------------------------------------
# 1. 파일 적재 (Excel to DataFrame)
# -------------------------------------------------------------------------
# 보직 분류 데이터셋(Excel)을 읽어옵니다.
soldier = pd.read_excel(p_xlsx)

# -------------------------------------------------------------------------
# 2. 분석 대상 컬럼 설정 및 행렬 변환
# -------------------------------------------------------------------------
# 역량 평가와 관련된 수치형 컬럼 리스트
numeric_skill_cols = [
    "Physical_Fitness",      # 체력 점수
    "Tech_Literacy",         # 기술 이해도
    "Suitability_Score",     # 적성 점수
    "Teamwork_Score",        # 협동심 점수
    "Mission_Success_Rate",  # 임무 성공률
    "Training_Performance",  # 훈련 성과
]

# 데이터프레임 내에 실제 존재하는 컬럼만 필터링
present = [c for c in numeric_skill_cols if c in soldier.columns]

# 고속 연산을 위해 해당 컬럼 데이터만 추출하여 NumPy ndarray(float)로 변환
X = soldier[present].to_numpy(dtype=float)

# -------------------------------------------------------------------------
# 3. NumPy 벡터화를 이용한 Z-score 표준화 (Standardization)
# -------------------------------------------------------------------------
# 각 역량 항목별 평균(mu)과 표준편차(sigma) 계산 (결측치 제외)
mu = np.nanmean(X, axis=0)
sigma = np.nanstd(X, axis=0)

# 표준편차가 0인 경우(모든 값이 동일) 분모가 0이 되는 것을 방지하기 위해 1.0으로 대체
sigma_safe = np.where(sigma < 1e-9, 1.0, sigma)

# Z-score 계산: (요소 - 평균) / 표준편차
# 이를 통해 서로 다른 척도의 점수들을 '평균 0, 표준편차 1'의 동일 선상에서 비교 가능
Z = (X - mu) / sigma_safe

# -------------------------------------------------------------------------
# 4. 종합 역량 지표(Index) 생성 및 저장
# -------------------------------------------------------------------------
# 표준화된 점수(z_컬럼명)를 원래 데이터프레임에 추가
soldier[[f"z_{c}" for c in present]] = Z

# [종합 지표 1: 평균 역량] 모든 표준화 점수의 평균 (전반적인 우수성)
soldier["capability_index_mean_z"] = np.nanmean(Z, axis=1)

# [종합 지표 2: 강점 합계] 0보다 큰(평균 이상인) 표준화 점수들만 합산 
# 어떤 병사가 특정 분야에서 '특출난 강점'을 가졌는지 파악하는 지표
soldier["capability_index_sum_pos"] = np.nansum(np.maximum(Z, 0), axis=1)

# -------------------------------------------------------------------------
# 5. 결과 확인 및 통계 요약
# -------------------------------------------------------------------------
# 원본 역량 점수들과 새롭게 생성된 '종합 역량 지표(평균)'의 분포 확인
soldier[present + ["capability_index_mean_z"]].describe()

,Physical_Fitness,Tech_Literacy,Suitability_Score,Teamwork_Score,Mission_Success_Rate,Training_Performance,capability_index_mean_z
count,22000.000000,22000.000000,22000.000000,22000.000000,22000.000000,22000.000000,2.200000e+04
mean,5.706182,0.284699,6.350818,7.006505,0.666521,6.504691,-7.848267e-17
std,2.494949,0.158618,2.216542,1.466865,0.178723,1.205207,4.075037e-01
min,1.000000,0.001000,0.000000,0.400000,0.040000,1.300000,-1.562883e+00
25%,4.000000,0.161000,5.000000,6.000000,0.540000,5.700000,-2.716583e-01
50%,6.000000,0.264000,6.000000,7.000000,0.690000,6.500000,2.241906e-04
75%,8.000000,0.388000,8.000000,8.000000,0.810000,7.300000,2.761969e-01
max,10.000000,0.870000,10.000000,10.000000,1.000000,10.000000,1.604554e+00


## (부록) 루프 vs 벡터화

동일한 계산을 **파이썬 루프**와 **`numpy`** 로 비교합니다. 행 수가 커질수록 차이가 큽니다.


In [6]:
# -------------------------------------------------------------------------
# 1. 난수 생성기 및 대규모 데이터 준비
# -------------------------------------------------------------------------
# 시드(Seed)를 0으로 고정하여 실행 시마다 동일한 결과를 얻는 Generator 생성
rng = np.random.default_rng(0)

n = 50_000  # 5만 개의 요소를 가진 배열 생성
# 0~1 사이의 난수를 생성하고 64비트 부동소수점으로 타입 지정
a = rng.random(n).astype(np.float64)
b = rng.random(n).astype(np.float64)

# -------------------------------------------------------------------------
# 2. 일반적인 For Loop를 이용한 연산 (순차 처리)
# -------------------------------------------------------------------------
t0 = time.perf_counter()  # 고정밀 타이머 시작
out_loop = np.empty(n)    # 결과를 담을 빈 배열 미리 생성

for i in range(n):
    # 각 인덱스에 하나씩 접근하여 계산 (Python 인터프리터 수준에서 반복)
    out_loop[i] = a[i] * b[i] + np.sqrt(a[i])

t_loop = time.perf_counter() - t0  # 루프 방식 소요 시간 측정

# -------------------------------------------------------------------------
# 3. NumPy 벡터화 연산 (병렬/일괄 처리)
# -------------------------------------------------------------------------
t0 = time.perf_counter()

# 배열 전체를 하나의 단위로 보고 연산 (내부적으로 C/Fortran 루프로 최적화되어 실행)
out_vec = a * b + np.sqrt(a)

t_vec = time.perf_counter() - t0  # 벡터화 방식 소요 시간 측정

# -------------------------------------------------------------------------
# 4. 결과 검증 및 성능 리포트
# -------------------------------------------------------------------------
# 두 방식의 계산 결과가 부동소수점 오차 범위(rtol=1e-12) 내에서 일치하는지 확인
np.testing.assert_allclose(out_loop, out_vec, rtol=1e-12)

print(f"loop:    {t_loop*1000:.1f} ms")   # 루프 방식 시간 (밀리초)
print(f"vector: {t_vec*1000:.1f} ms")    # 벡터화 방식 시간 (밀리초)
print(f"speedup: {t_loop / t_vec:.1f}x")  # 벡터화가 얼마나 더 빠른지 배수 출력

loop:    17.3 ms
vector: 0.4 ms
speedup: 39.4x


## 요약

| 데이터 | 주요 벡터화 기법 | 파생변수 예 |
|--------|------------------|-------------|
| 군수품 | 열 간 사칙·`clip` | `stock_runway_days`, `expected_demand_in_lead_time` |
| 서버 | 축 집계 `mean(axis=1)`, `log1p` | `resource_load_mean`, `log1p_fix_hours` |
| 복지 | 브로드캐스팅, 행렬 min-max | `access_burden_index`, `*_01` |
| Excel | 열 방향 `mean`/`std`로 z-score | `z_*`, `capability_index_mean_z` |

이후 **학습·평가** 단계에서는 위에서 만든 열을 `DataFrame` 그대로 쓰거나 `.to_numpy()` 로 행렬을 뽑아 모델에 넣으면 됩니다.
